In [0]:
%pip install cloudpickle mlflow  catboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================
# SENTIMENT ANALYSIS - MODEL SERVING NOTEBOOK
# Purpose: Register model in Unity Catalog and create Serving Endpoint
# ============================================

import mlflow
import cloudpickle
import pandas as pd
import mlflow.pyfunc
from mlflow.models import infer_signature
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------
# 1. DEFINE THE SENTIMENT PIPELINE CLASS
# -------------------------------------------------
class AspectSentimentPipeline:
    def __init__(self, model_name="typeform/distilbert-base-uncased-mnli"):
        self.classifier = pipeline("zero-shot-classification", model=model_name)
        self.aspects = ["delivery", "product quality", "price", "customer support", "packaging"]
        self.sentiment_labels = ["positive", "neutral", "negative"]
    
    def analyze_single(self, feedback):
        """Analyze a single feedback string. Returns dict {aspect: sentiment}."""
        if not isinstance(feedback, str) or len(feedback.strip()) == 0:
            return {}
        
        # Step 1: Detect which aspects are present
        aspect_result = self.classifier(feedback, self.aspects, multi_label=True)
        detected = [label for label, score in zip(aspect_result['labels'], aspect_result['scores']) if score > 0.5]
        
        # Step 2: For each detected aspect, get sentiment
        result = {}
        for asp in detected:
            template = "The sentiment regarding {} is"
            sent_result = self.classifier(feedback, self.sentiment_labels, hypothesis_template=template)
            result[asp] = sent_result['labels'][0]
        return result

# -------------------------------------------------
# 2. CREATE PYFUNC WRAPPER FOR MLFLOW
# -------------------------------------------------
class SentimentWrapper(mlflow.pyfunc.PythonModel):
    """Wrapper to make the pipeline compatible with MLflow Model Serving."""
    
    def load_context(self, context):
        """Load the pipeline from artifacts when the model is deployed."""
        import cloudpickle
        with open(context.artifacts["pipeline_path"], "rb") as f:
            self.pipeline = cloudpickle.load(f)
    
    def predict(self, context, model_input):
        """Predict sentiment for each input text."""
        # model_input is a pandas DataFrame with column 'text'
        return model_input['text'].apply(self.pipeline.analyze_single).tolist()

# -------------------------------------------------
# 3. LOAD THE TRAINED PIPELINE
# -------------------------------------------------
# Path to your saved pipeline in Unity Catalog Volume
VOLUME_PATH = "/Volumes/workspace/default/ml_artifacts/"
PIKLE_PATH = VOLUME_PATH + "aspect_sentiment_pipeline.pkl"

# Load the pipeline
with open(PIKLE_PATH, "rb") as f:
    pipeline = cloudpickle.load(f)

print("✅ Pipeline loaded successfully")

# -------------------------------------------------
# 4. CREATE SAMPLE INPUT AND OUTPUT FOR SIGNATURE
# -------------------------------------------------
sample_input = pd.DataFrame({"text": ["Delivery was very slow, but product quality is great."]})

# Create a temporary wrapper to generate sample output
temp_wrapper = SentimentWrapper()
temp_wrapper.pipeline = pipeline
sample_output = temp_wrapper.predict(None, sample_input)

# Infer the signature (required for Unity Catalog)
signature = infer_signature(sample_input, sample_output)

print("✅ Signature created")

# -------------------------------------------------
# 5. SAVE MODEL AS MLFLOW PYFUNC MODEL
# -------------------------------------------------
model_path = "/tmp/sentiment_model_final"

mlflow.pyfunc.save_model(
    path=model_path,
    python_model=SentimentWrapper(),
    artifacts={"pipeline_path": PIKLE_PATH},
    signature=signature,
    input_example=sample_input
)

print("✅ Model saved successfully at:", model_path)

# -------------------------------------------------
# 6. REGISTER MODEL IN UNITY CATALOG
# -------------------------------------------------
mlflow.set_registry_uri("databricks-uc")

registered_model = mlflow.register_model(
    model_uri=model_path,
    name="workspace.default.sentiment_analysis_model"
)

print(f"✅ Model registered in Unity Catalog as version {registered_model.version}")